In [ ]:
import sys
sys.path.append('./src')

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import acf
import matplotlib.pyplot as plt
from statsforecast import StatsForecast
from statsforecast.models import Naive, SeasonalNaive, AutoTheta, AutoETS
from statsforecast.utils import AirPassengersDF
from pathlib import Path
from data import load_tsf_robust, load_selected_series
from features import create_lag_features, create_fourier_features, create_calendar_features
from metrics import calculate_metrics

Загрузка

In [2]:
from series import load_tsf_robust

df = load_tsf_robust('m4_monthly_dataset.tsf')
print(f"Загружено {df['unique_id'].nunique()} рядов")
print(df.head())

Загружено 48000 рядов
  unique_id  ds       y
0        M1   0  8000.0
1        M1   1  8350.0
2        M1   2  8570.0
3        M1   3  7700.0
4        M1   4  7080.0


In [3]:
import json
from series import load_selected_series

saved_ids = load_selected_series()
final_series_ids = saved_ids

In [4]:
experiment_df = df[df['unique_id'].isin(final_series_ids)].copy()

In [7]:
np.random.seed(42)
sample_ids = np.random.choice(df['unique_id'].unique(), size=200, replace=False)
sample_df = df[df['unique_id'].isin(sample_ids)].copy()

In [ ]:
import plotly.graph_objects as go

np.random.seed(42)
top_10_ids = final_series_ids[:10]

fig = go.Figure()

for uid in top_10_ids:
    series_data = sample_df[sample_df['unique_id'] == uid]
    fig.add_trace(
        go.Scatter(
            x=series_data['ds'], 
            y=series_data['y'], 
            name=str(uid),
            mode='lines',
            line=dict(width=1)
        )
    )

fig.update_layout(
    title="Примеры рядов с сезонностью",
    xaxis_title="Время",
    yaxis_title="Значение",
    height=500,
    width=1000,
    template='plotly_white'
)

fig.show()

In [9]:
Path('./results/figures').mkdir(parents=True, exist_ok=True)

HORIZON = 18
SEASONAL_LAG = 12

sf_df = experiment_df.copy()
sf_df['ds'] = sf_df['ds'].astype(int) + 1

print(f"Рядов: {sf_df['unique_id'].nunique()}, горизонт: {HORIZON}")

Рядов: 100, горизонт: 18


Бейзлайны

In [11]:
sf = StatsForecast(
    models=[
        Naive(),
        SeasonalNaive(season_length=SEASONAL_LAG),
        AutoTheta(),
        AutoETS(model='ZZZ')
    ],
    freq=1
)

forecasts = sf.forecast(df=sf_df, h=HORIZON, fitted=True)
print(forecasts.head())

  unique_id   ds   Naive  SeasonalNaive    AutoTheta      AutoETS
0    M10158  131  1784.0         1775.0  1796.249980  1803.849175
1    M10158  132  1784.0         1776.0  1798.466158  1806.620790
2    M10158  133  1784.0         1781.0  1800.682335  1809.392405
3    M10158  134  1784.0         1790.0  1802.898512  1812.164020
4    M10158  135  1784.0         1789.0  1805.114690  1814.935635


Делаем train\test split

In [12]:
def create_train_test_split(df, horizon, unique_ids):
    train_list = []
    test_list = []
    
    for uid in unique_ids:
        series = df[df['unique_id'] == uid].copy()
        if len(series) <= horizon:
            continue
            
        train = series.iloc[:-horizon].copy()
        test = series.iloc[-horizon:].copy()
        
        train_list.append(train)
        test_list.append(test)
    
    return pd.concat(train_list), pd.concat(test_list)

train_df, test_df = create_train_test_split(experiment_df, HORIZON, final_series_ids)
print(f"Разбиение: train {len(train_df)}, test {len(test_df)}")

Разбиение: train 23147, test 1800


In [13]:
sf_train = StatsForecast(
    models=[Naive(), SeasonalNaive(season_length=SEASONAL_LAG), AutoTheta(), AutoETS(model='ZZZ')],
    freq=1
)

baseline_forecasts = sf_train.forecast(df=train_df, h=HORIZON)

In [14]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
from metrics import calculate_metrics

results_df = test_df.merge(baseline_forecasts, on=['unique_id', 'ds'], how='left')

metrics_results = {}
for model in ['Naive', 'SeasonalNaive', 'AutoTheta', 'AutoETS']:
    col_name = model if model != 'SeasonalNaive' else 'SeasonalNaive'
    if col_name in results_df.columns:
        metrics_results[model] = calculate_metrics(
            results_df['y'].values,
            results_df[col_name].values
        )

metrics_df = pd.DataFrame(metrics_results).T
print("\nМетрики базовых моделей:")
print(metrics_df.round(4))


Метрики базовых моделей:
                    MAE      RMSE     MAPE
Naive          447.2859  782.3140  14.8165
SeasonalNaive  443.1410  733.0858  16.0398
AutoTheta      400.6173  724.5381  12.2296
AutoETS        444.0067  766.1434  14.5369


Вычислим силу сезонности для выбранных рядов

In [15]:
def get_seasonality_strength(series, seasonal_lag=12):
    if isinstance(series, np.ndarray):
        series = pd.Series(series)
    if len(series.dropna()) < seasonal_lag * 2:
        return 0
    acf_values = acf(series.dropna(), nlags=seasonal_lag * 2, fft=False)
    return acf_values[seasonal_lag]

seasonality_strength = {}
for uid in final_series_ids:
    series = sample_df[sample_df['unique_id'] == uid]['y'].values
    seasonality_strength[uid] = get_seasonality_strength(series, SEASONAL_LAG)

weak_seasonality = [uid for uid, s in seasonality_strength.items() if s < 0.4]
strong_seasonality = [uid for uid, s in seasonality_strength.items() if s >= 0.4]

print(f"Слабая сезонность: {len(weak_seasonality)} рядов")
print(f"Сильная сезонность: {len(strong_seasonality)} рядов")

Слабая сезонность: 6 рядов
Сильная сезонность: 94 рядов


In [16]:
from catboost import CatBoostRegressor
from feature_generation import create_lag_features, create_fourier_features, create_calendar_features

full_df = experiment_df.copy()

full_df = create_lag_features(full_df, lags=[1, 2, 3, 4, 5, 6, 12, 24, 36])
full_df = create_fourier_features(full_df, seasonal_period=SEASONAL_LAG, n_harmonics=3)
full_df = create_calendar_features(full_df)

train_list = []
test_list = []

for uid in full_df['unique_id'].unique():
    series = full_df[full_df['unique_id'] == uid].sort_values('ds')
    if len(series) <= HORIZON:
        continue
    train = series.iloc[:-HORIZON].copy()
    test = series.iloc[-HORIZON:].copy()
    train_list.append(train)
    test_list.append(test)

train_feat = pd.concat(train_list)
test_feat = pd.concat(test_list)

train_feat = train_feat.dropna()
test_feat = test_feat.dropna()

print(f"Train: {len(train_feat)}, Test: {len(test_feat)}")

Train: 19547, Test: 1800


In [17]:
FEATURE_SETS = {
    'regular': [f'lag_{i}' for i in [1, 2, 3, 4, 5, 6]],
    'seasonal': [f'lag_{i}' for i in [12, 24, 36]],
    'fourier': [f'fourier_sin_{i}' for i in [1, 2, 3]] + [f'fourier_cos_{i}' for i in [1, 2, 3]],
    'calendar': ['month', 'quarter'],
    'combined_1': [f'lag_{i}' for i in [1, 2, 3, 4, 5, 6, 12, 24, 36]],
    'combined_2': [f'lag_{i}' for i in [1, 2, 3, 4, 5, 6, 12, 24, 36]] + 
                  [f'fourier_sin_{i}' for i in [1, 2, 3]] + 
                  [f'fourier_cos_{i}' for i in [1, 2, 3]] + 
                  ['month', 'quarter']
}

In [ ]:
all_results = {}

for name, features in FEATURE_SETS.items():
    valid_features = [f for f in features if f in train_feat.columns]
    if not valid_features:
        continue
    
    X_train = train_feat[valid_features].values
    y_train = train_feat['y'].values
    X_test = test_feat[valid_features].values
    y_test = test_feat['y'].values
    
    
    model = CatBoostRegressor(
        iterations=500,
        loss_function='MAE',
        eval_metric='MAE',
        verbose=False,
        random_seed=42
    )
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    mask = y_test != 0
    mape = np.mean(np.abs((y_test[mask] - predictions[mask]) / y_test[mask])) * 100
    
    all_results[name] = {'MAE': mae, 'RMSE': rmse, 'MAPE': mape}
    print(f"{name}: MAE={mae:}, RMSE={rmse:}, MAPE={mape:}")

regular: MAE=271.52, RMSE=574.32, MAPE=6.82
seasonal: MAE=396.92, RMSE=641.91, MAPE=14.04
fourier: MAE=2295.40, RMSE=3147.31, MAPE=77.87
calendar: MAE=2386.75, RMSE=3307.07, MAPE=81.28
combined_1: MAE=242.48, RMSE=427.50, MAPE=6.82
combined_2: MAE=240.07, RMSE=434.10, MAPE=6.67


In [19]:
results_catboost = pd.DataFrame(all_results).T
print("\nРезультаты CatBoost разных наборов признаков:")
print(results_catboost.round(4))


Результаты CatBoost разных наборов признаков:
                  MAE       RMSE     MAPE
regular      271.5170   574.3206   6.8167
seasonal     396.9198   641.9074  14.0438
fourier     2295.3964  3147.3111  77.8685
calendar    2386.7535  3307.0671  81.2836
combined_1   242.4850   427.5035   6.8249
combined_2   240.0728   434.0972   6.6698


Проверяем эффект признаков отдельно для рядов с сильной и слабой сезонностью

In [21]:
def test_seasonality_dependence(df_full, train_df, test_df, seasonality_groups, horizon, seasonal_lag):
    results_by_strength = {}
    
    for group_name, group_ids in seasonality_groups.items():
        group_results = {}
        
        train_group = train_df[train_df['unique_id'].isin(group_ids)]
        test_group = test_df[test_df['unique_id'].isin(group_ids)]
        
        for name, features in FEATURE_SETS.items():
            train_subset = train_feat[train_feat['unique_id'].isin(group_ids)]
            test_subset = test_feat[test_feat['unique_id'].isin(group_ids)]
            
            valid_features = [f for f in features if f in train_subset.columns]
            
            X_train = train_subset[valid_features].values
            y_train = train_subset['y'].values
            X_test = test_subset[valid_features].values
            y_test = test_subset['y'].values
            
            model = CatBoostRegressor(
                iterations=500,
                loss_function='MAE',
                eval_metric='MAE',
                verbose=False,
                random_seed=42
            )
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)
            
            mae = mean_absolute_error(y_test, predictions)
            group_results[name] = mae
        
        results_by_strength[group_name] = group_results
    
    return results_by_strength

seasonality_groups = {
    'Слабая сезонность': weak_seasonality,
    'Сильная сезонность': strong_seasonality
}

dependence = test_seasonality_dependence(
    full_df, train_df, test_df, seasonality_groups, HORIZON, SEASONAL_LAG
)

print("\nРезультаты по силе сезонности (MAE):")
for group, results in dependence.items():
    print(f"\n{group}:")
    for feature_set, mae in sorted(results.items(), key=lambda x: x[1]):
        print(f"  {feature_set}: {mae:.2f}")


Результаты по силе сезонности (MAE):

Слабая сезонность:
  combined_1: 213.12
  combined_2: 223.91
  regular: 225.70
  seasonal: 431.27
  fourier: 691.24
  calendar: 796.20

Сильная сезонность:
  combined_2: 242.87
  combined_1: 244.31
  regular: 276.47
  seasonal: 382.23
  fourier: 2388.15
  calendar: 2472.98


In [24]:
final_comparison = metrics_df.copy()
final_comparison.loc['CatBoost_Best'] = {
    'MAE': 240.18,
    'RMSE': 434.0972,
    'MAPE': 6.6698
}

final_comparison = final_comparison.sort_values('MAE')
print("\nСравнение моделей:")
print(final_comparison.round(4))

best_baseline = metrics_df.loc['AutoTheta', 'MAE']
catboost_mae = 240.18
improvement = ((best_baseline - catboost_mae) / best_baseline) * 100



Сравнение моделей:
                    MAE      RMSE     MAPE
CatBoost_Best  240.1800  434.0972   6.6698
AutoTheta      400.6173  724.5381  12.2296
SeasonalNaive  443.1410  733.0858  16.0398
AutoETS        444.0067  766.1434  14.5369
Naive          447.2859  782.3140  14.8165


Тестирует влияние горизонта прогнозирования на качество CatBoost

In [29]:
def test_multiple_horizons(df_exp, feature_sets, horizons=[6, 12, 18, 24]):
    horizon_results = {}
    
    for horizon in horizons:
        train_list = []
        test_list = []
        
        for uid in df_exp['unique_id'].unique():
            series = df_exp[df_exp['unique_id'] == uid].copy()
            if len(series) <= horizon:
                continue
            
            train = series.iloc[:-horizon].copy()
            test = series.iloc[-horizon:].copy()
            
            train_list.append(train)
            test_list.append(test)
        
        train_tmp = pd.concat(train_list, ignore_index=False)
        test_tmp = pd.concat(test_list, ignore_index=False)
        
        df_with_lags = create_lag_features(train_tmp, lags=[1, 2, 3, 4, 5, 6, 12, 24, 36])
        df_with_lags = create_fourier_features(df_with_lags, seasonal_period=SEASONAL_LAG, n_harmonics=3)
        df_with_lags = create_calendar_features(df_with_lags)
        
        train_feat_tmp = df_with_lags.dropna()
        

        test_with_features = test_tmp.copy()
        test_with_features = create_fourier_features(test_with_features, seasonal_period=SEASONAL_LAG, n_harmonics=3)
        test_with_features = create_calendar_features(test_with_features)
        test_feat_tmp = test_with_features.dropna()
        
        print(f"  Train: {len(train_feat_tmp)}, Test: {len(test_feat_tmp)}")
        
        horizon_mae = {}
        
        test_sets = ['regular', 'seasonal', 'fourier', 'calendar', 'combined_2']
        
        for name in test_sets:
            features = feature_sets[name]
            valid_features = [f for f in features if f in train_feat_tmp.columns and f in test_feat_tmp.columns]
            
            if not valid_features or len(test_feat_tmp) == 0:
                continue
            
            try:
                X_train = train_feat_tmp[valid_features].values
                y_train = train_feat_tmp['y'].values
                X_test = test_feat_tmp[valid_features].values
                y_test = test_feat_tmp['y'].values
                
                if len(X_test) == 0 or len(y_test) == 0:
                    continue
                
                model = CatBoostRegressor(
                    iterations=300,
                    loss_function='MAE',
                    eval_metric='MAE',
                    verbose=False,
                    random_seed=42
                )
                model.fit(X_train, y_train)
                predictions = model.predict(X_test)
                
                mae = mean_absolute_error(y_test, predictions)
                horizon_mae[name] = mae
                print(f"  {name}: MAE={mae:.2f}")
            except Exception as e:
                print(f"  {name}: ошибка - {e}")
                continue
        
        if horizon_mae:
            horizon_results[horizon] = horizon_mae

    
    return horizon_results

horizon_analysis = test_multiple_horizons(experiment_df, FEATURE_SETS)

print("\n\nРезультаты по горизонтам (MAE):")
if horizon_analysis:
    horizon_df = pd.DataFrame(horizon_analysis).T
    print(horizon_df.round(2))

  Train: 20747, Test: 600
  fourier: MAE=2371.78
  calendar: MAE=2431.45
  combined_2: MAE=2375.38
  Train: 20147, Test: 1200
  fourier: MAE=2320.61
  calendar: MAE=2395.80
  combined_2: MAE=2322.77
  Train: 19547, Test: 1800
  fourier: MAE=2310.43
  calendar: MAE=2386.81
  combined_2: MAE=2312.66
  Train: 18947, Test: 2400
  fourier: MAE=2283.90
  calendar: MAE=2352.28
  combined_2: MAE=2288.46


Результаты по горизонтам (MAE):
    fourier  calendar  combined_2
6   2371.78   2431.45     2375.38
12  2320.61   2395.80     2322.77
18  2310.43   2386.81     2312.66
24  2283.90   2352.28     2288.46
